In [1]:
!pip install transformers accelerate -q

In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
model.eval()
print('Model loaded!')

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [3]:
def generate(prompt, max_new_tokens=150, temperature=0.9, top_k=50, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
prompts = [
    "Artificial intelligence is transforming",
    "Deep learning models can",
    "The transformer architecture",
]

for prompt in prompts:
    print(f'\nPrompt: {prompt}')
    print('-' * 50)
    print(generate(prompt))


Prompt: Artificial intelligence is transforming
--------------------------------------------------
Artificial intelligence is transforming our lives," said Michael A. Borenstein, a former chairman of the Office of Management and Budget who now teaches at Harvard Law School
"It makes it easier for government to provide better benefits than private companies."

Prompt: Deep learning models can
--------------------------------------------------
Deep learning models can lead to more interesting, personalized approaches that will be helpful for people who find themselves on the wrong side of a complex problem.


: "The next big game in this room is AI," said Mark Heisler , an analyst at CSPI and co-author of The Next Generation Lab : Intelligent Machine Learning Is Being Made More Easier By Our Automated Minds . It's also one reason why artificial intelligence (AI) may ultimately become such fundamental technologies from which our very selves are drawn when it comes time after work or fami

In [5]:
prompt = "The future of machine learning"
inputs = tokenizer(prompt, return_tensors='pt').to(device)

strategies = {
    'Greedy':        dict(do_sample=False),
    'Beam Search':   dict(do_sample=False, num_beams=4),
    'Top-K':         dict(do_sample=True, top_k=50, temperature=1.0),
    'Top-P':         dict(do_sample=True, top_p=0.95, top_k=0, temperature=1.0),
    'Top-K + Top-P': dict(do_sample=True, top_k=50, top_p=0.92, temperature=0.85),
}

for name, kwargs in strategies.items():
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=60,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    print(f'\n--- {name} ---')
    print(tokenizer.decode(out[0], skip_special_tokens=True))


--- Greedy ---
The future of machine learning is in the hands of the next generation of machine learning researchers.

Machine learning is a new field that has been around for a while. It is a field that has been around for a while. It is a field that has been around for a while. It is a field that has been

--- Beam Search ---
The future of machine learning

Machine learning has been around for a long time, and it's been used in a number of different fields, including machine learning, machine learning, machine learning, machine learning, machine learning, machine learning, machine learning, machine learning, machine learning, machine learning, machine learning, machine learning

--- Top-K ---
The future of machine learning may seem far from obvious to us. But we can't ignore how important it is.

Machine learning, which is a field of research where we measure what is learned, can be a very powerful tool to understand artificial intelligence from a number of directions. There are two

In [6]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import Dataset

training_text = """
Artificial intelligence is the simulation of human intelligence by machines.
Machine learning is a subset of AI that learns from data automatically.
Deep learning uses neural networks with many layers to learn representations.
Natural language processing allows computers to understand human language.
The transformer architecture uses attention mechanisms for sequence modeling.
GPT-2 is a large language model trained to predict the next word in text.
Fine-tuning adapts a pre-trained model to a specific domain or writing style.
Transfer learning reuses knowledge from one task to help with another task.
Tokenization splits text into smaller units called tokens for processing.
Language models assign probabilities to sequences of words or tokens.
Reinforcement learning trains agents to take actions that maximize rewards.
Computer vision enables machines to interpret and understand visual data.
Generative models can create new data samples similar to the training data.
Attention mechanisms allow models to focus on relevant parts of the input.
Neural networks are inspired by the structure of the human brain.
"""

class SimpleDataset(Dataset):
    def __init__(self, tokenizer, text, block_size=64):
        tokenized = tokenizer(text, return_tensors='pt',
                             truncation=False,
                             add_special_tokens=False)['input_ids'][0]
        self.examples = [tokenized[i:i+block_size]
                        for i in range(0, len(tokenized)-block_size+1, block_size)]
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i].clone().detach()

ft_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_model = GPT2LMHeadModel.from_pretrained('gpt2')

dataset = SimpleDataset(ft_tokenizer, training_text)
print(f'Training blocks: {len(dataset)}')

training_args = TrainingArguments(
    output_dir='./gpt2-finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=5,
    report_to='none',
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(ft_tokenizer, mlm=False),
    train_dataset=dataset,
)

trainer.train()
print('Fine-tuning complete!')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Training blocks: 3


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.598036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete!


In [7]:
ft_model.eval()

test_prompts = [
    'Artificial intelligence is',
    'Deep learning uses',
    'The transformer',
]

for prompt in test_prompts:
    inputs = ft_tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.85,
            top_k=50,
            repetition_penalty=1.2,
            pad_token_id=ft_tokenizer.eos_token_id,
        )
    print(f'\nPrompt: {prompt}')
    print('-' * 40)
    print(ft_tokenizer.decode(out[0], skip_special_tokens=True))


Prompt: Artificial intelligence is
----------------------------------------
Artificial intelligence is an exciting field to study. The question of whether artificial features can work or not, and what kind will be considered "normal," remains a matter of controversy due in no small part (although this debate has expanded recently across social technologies) into the realm that AI-driven machines do play such roles as regulators for government agencies like education departments... But right now we are seeing quite distinct technological developments from those

Prompt: Deep learning uses
----------------------------------------
Deep learning uses a set of tricks that help you learn and create solutions.
The language for teaching can be written by different people, or it will have some features like automatic naming recognition (also called implicit understanding) so your code won't break if something breaks before reading the text file with just one input line from another person's pr